========================================================= <br>
Spark Certification Study Notebook Gold Layer — Analytical Data Models
========================================================= <br>

# Import Environment

In [0]:
# Import environment variables
from setup.env import *

from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark.conf.set("spark.sql.shuffle.partitions", DEFAULT_SHUFFLE_PARTITIONS)

# Gold Layer — Analytical Data Models

<b> Goal </br>

The Gold layer represents the business-ready analytical layer of the Lakehouse.

In this layer we create curated datasets optimized for:
- business analytics
- dashboards
- reporting
- high-level KPIs

Gold tables usually contain:
- aggregated metrics
- denormalized datasets
- optimized queries

<b>Input Tables (Silver Layer)</b>

We will read the cleaned datasets created in the Silver layer.

Tables available:
- spark_cert_catalog.silver.customers
- spark_cert_catalog.silver.products
- spark_cert_catalog.silver.orders
- spark_cert_catalog.silver.order_items

<b> Output Views (Gold Layer)</b>

We will create two analytical views:
| View                | Purpose                                      |
| ------------------- | -------------------------------------------- |
| sales_metrics_exact | Aggregations using exact functions           |
| sales_metrics_hll   | Aggregations using HyperLogLog approximation |


## TASK 1 — Load Silver Tables

<b> Why </b>

The Gold layer should never read raw data.

It always reads curated Silver datasets to guarantee:
- data quality
- consistent schemas
- reliable analytics

<b> Your Task </b></br>
Load the following Silver tables:
- customers
- products
- orders
- order_items

Store them in the following DataFrames:
- customers_silver_df
- products_silver_df
- orders_silver_df
- order_items_silver_df

### Your solution

### Suggested solution

In [0]:
print(50*"=")
print("Starting Gold Layer: Loading Silver Tables")

In [0]:
customers_silver_df = spark.table(f"{CATALOG_NAME}.{SCHEMA_SILVER}.customers")
products_silver_df = spark.table(f"{CATALOG_NAME}.{SCHEMA_SILVER}.products")
orders_silver_df = spark.table(f"{CATALOG_NAME}.{SCHEMA_SILVER}.orders")
order_items_silver_df = spark.table(f"{CATALOG_NAME}.{SCHEMA_SILVER}.order_items")

## TASK 2 — Create Exact Aggregation Metrics

<b> Why </b>

Traditional aggregations provide exact results, but they can become expensive for large datasets.

Common exact aggregations include:
- count
- sum
- avg
- countDistinct

These are useful for accurate financial and business reporting.

Metrics to Calculate

Group by:
- country
- category

Metrics:
| Metric           | Function      |
| ---------------- | ------------- |
| total_orders     | count         |
| total_revenue    | sum           |
| avg_order_value  | avg           |
| unique_customers | countDistinct |

<b> Your Task </b>

Create a DataFrame named:
```
sales_metrics_exact_df
```

### Your solution

### Suggested solution

In [0]:
from pyspark.sql.functions import *

sales_metrics_exact_df = (
    order_items_silver_df.alias('order_itens')
    .join(orders_silver_df, "order_id")
    .join(customers_silver_df, "customer_id")
    .join(products_silver_df, "product_id")
    .groupBy("country", "category")
    .agg(
        count("order_id").alias("total_orders"),
        sum("order_itens.price").alias("total_revenue"),
        avg("order_itens.price").alias("avg_order_value"),
        countDistinct("customer_id").alias("unique_customers")
    )
)

sales_metrics_exact_df.limit(5).display()

### TASK 3 — Create HyperLogLog Approximation Metrics

<b> Why </b>

For very large datasets, exact distinct counting becomes expensive.

Spark provides HyperLogLog-based estimation through:
```
approx_count_distinct()
```

HyperLogLog:
- uses probabilistic algorithms
- consumes much less memory
- scales better for big data

Tradeoff:
- Small error margin
- but much faster performance

<b>Metrics to Calculate</b></br>
We will compute the same metrics as the exact version, but replace:
```
countDistinct()
```

with
```
approx_count_distinct()
```

<b> Your Task </b></br>
Create a DataFrame named:
- sales_metrics_hll_df

### Your solution

### Suggested solution

In [0]:
sales_metrics_hll_df = (
    order_items_silver_df.alias("order_itens")
    .join(orders_silver_df, "order_id")
    .join(customers_silver_df, "customer_id")
    .join(products_silver_df, "product_id")
    .groupBy("country", "category")
    .agg(
        count("order_id").alias("total_orders"),
        sum("order_itens.price").alias("total_revenue"),
        avg("order_itens.price").alias("avg_order_value"),
        approx_count_distinct("customer_id").alias("unique_customers_estimated")
    )
)

sales_metrics_hll_df.limit(5).display()


## TASK 4 — Window Functions for Analytical Insights

<b> Why </b>

Window functions allow calculations across rows without collapsing the dataset.

Unlike groupBy, window functions retain row-level data.

Common use cases:
- ranking
- cumulative metrics
- trend analysis
- time-series analytics

<b> Window Function Concepts </b>
```
OVER (
    PARTITION BY ...
    ORDER BY ...
)
```
<b> Example: <b>
- ROW_NUMBER()
- RANK()
- DENSE_RANK()
- LAG()
- LEAD()

<b> Your Task </b>

Create analytical metrics including:
| Metric                  | Function   |
| ----------------------- | ---------- |
| product_rank_by_revenue | rank       |
| customer_order_sequence | row_number |
| previous_order_value    | lag        |
| next_order_value        | lead       |


### Your solution

### Suggested solution

In [0]:
from pyspark.sql.window import Window

sales_df = (
    order_items_silver_df.alias('order_itens')
    .join(orders_silver_df, "order_id")
    .join(products_silver_df, "product_id")
)

product_window = Window.partitionBy("category").orderBy(desc("order_itens.price"))

customer_window = Window.partitionBy("customer_id").orderBy("order_timestamp")

sales_window_df = (
    sales_df
    .withColumn(
        "product_rank_by_revenue",
        rank().over(product_window)
    )
    .withColumn(
        "customer_order_sequence",
        row_number().over(customer_window)
    )
    .withColumn(
        "previous_order_value",
        lag("order_itens.price", 1).over(customer_window)
    )
    .withColumn(
        "next_order_value",
        lead("order_itens.price", 1).over(customer_window)
    )
)

display(sales_window_df.limit(5))

### TASK 5 — Monthly Revenue

<b> Why </b>

Revenue analysis over time is one of the most common analytics workloads.

Monthly revenue metrics are used for:
- business growth analysis
- seasonality detection
- executive dashboards

Metrics to Compute

Group by:
```
year
month
```
Metrics:
| Metric           | Function      |
| ---------------- | ------------- |
| monthly_revenue  | sum           |
| total_orders     | count         |
| unique_customers | countDistinct |
| avg_order_value  | avg           |


### Your solution

### Suggested solution

In [0]:
from pyspark.sql.functions import *

sales_df = (
    order_items_silver_df.alias('order_itens')
    .join(orders_silver_df, "order_id")
    .join(customers_silver_df, "customer_id")
)

monthly_revenue_df = (
    sales_df
    .withColumn("year", year("order_timestamp"))
    .withColumn("month", month("order_timestamp"))
    .groupBy("year", "month")
    .agg(
        sum("order_itens.price").alias("monthly_revenue"),
        count("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers"),
        avg("order_itens.price").alias("avg_order_value")
    )
    .orderBy("year", "month")
)

monthly_revenue_df.limit(5).display()

## TASK 6 — Top Products

<b> Why </b>

Product ranking is a classic business intelligence metric.

It allows analysts to identify:
- best selling products
- high revenue categories
- product performance trends

Metrics

Group by:
```
product_id
product_name
category
```
Metrics:
| Metric           | Function      |
| ---------------- | ------------- |
| total_sales      | sum           |
| total_orders     | count         |
| unique_customers | countDistinct |
| avg_price        | avg           |


### Your solution

### Suggested solution

In [0]:

top_products_df = (
    order_items_silver_df.alias('order_itens')
    .join(orders_silver_df, "order_id")
    .join(products_silver_df, "product_id")
    .join(customers_silver_df, "customer_id")
    .groupBy("product_id", "name", "category")
    .agg(
        sum("order_itens.price").alias("total_sales"),
        count("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers"),
        avg("order_itens.price").alias("avg_price")
    )
)

display(top_products_df.limit(5))

## TASK 7 — Save Gold Layer as Materialized Views

<b> Why </b>

Gold datasets are often materialized for fast analytical queries.

Materializing datasets provides:
- faster BI queries
- reusable datasets
- simplified dashboards

Instead of recalculating transformations every query.

<b> Views to Materialize </b>

| View Name                |
| ------------------------ |
| gold_sales_metrics_exact |
| gold_sales_metrics_hll   |
| gold_monthly_revenue     |
| gold_top_products        |


Use: 
```
CREATE OR REPLACE MATERIALIZED VIEW
```

### Your solution

### Suggested solution

In [0]:
sales_metrics_exact_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_GOLD}.sales_metrics_exact")

sales_metrics_hll_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_GOLD}.sales_metrics_hll")

monthly_revenue_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_GOLD}.monthly_revenue")

top_products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_GOLD}.top_products")